# Assignment 3 Report - Residual Neural Network for classifying BloodMNIST images

### Tevyn Vergara | 46421201 | Electrical & Biomedical Engineer

### Note: I have approved extension until 18/05


#### DISCLAIMER: GPT-5.5 was used for this assignment for:
1. Layout tips
2. Syntax checking
3. Trouble shooting and debugging
4. Code and theory explanation
5. Re-organizing to look more modular and readable

In reference: [1]

#### References:
[1] OpenAI, “ChatGPT, GPT-5.5,” ChatGPT, May 18, 2026. [Online]. Available: https://chatgpt.com/

[2] J. Yang, R. Shi, D. Wei, Z. Liu, L. Zhao, B. Ke, H. Pfister, and B. Ni, “MedMNIST v2—A large-scale lightweight benchmark for 2D and 3D biomedical image classification,” Scientific Data, vol. 10, no. 1, Art. no. 41, Jan. 2023, doi: 10.1038/s41597-022-01721-8

[3] GeeksforGeeks (2025) Hyperparameter tuning: Fixing overfitting in Neural Networks, GeeksforGeeks. Available at: https://www.geeksforgeeks.org/machine-learning/hyperparameter-tuning-fixing-overfitting-in-neural-networks/ (Accessed: 18 May 2026). 

[4] D. P. Kingma and J. Ba, “Adam: A method for stochastic optimization,” *arXiv preprint arXiv:1412.6980*, 2014, doi: [10.48550/arXiv.1412.6980](https://doi.org/10.48550/arXiv.1412.6980).

[5] PyTorch, “CrossEntropyLoss,” *PyTorch 2.12 Documentation*. Accessed: May 20, 2026. [Online]. Available: [https://docs.pytorch.org/docs/2.12/generated/torch.nn.modules.loss.CrossEntropyLoss.html](https://docs.pytorch.org/docs/2.12/generated/torch.nn.modules.loss.CrossEntropyLoss.html)


#### Link for pre-trained models:
https://drive.google.com/drive/folders/105GiBqKmAd7X4MozeUwmY6MARaGfrcT1?usp=sharing 


## 1. The ResNet18 Neural Network Structure

The implemented model follows a ResNet18-style residual neural network to that used in [2], with the purpose of learning increasingly abstract image features from BloodMNIST blood cell images. 

- The initial convolution and pooling layers: early feature extraction and reduce spatial size. Residual block groups progressively increase the number of feature channels so that the network can learn more complex visual patterns. 

- The residual connections: allow information to pass through shortcut paths, helping the network train more reliably by reducing the risk of vanishing gradients in deeper layers. 

- The final pooling and fully connected layers: condense feature maps into class-level evidence for the 8 BloodMNIST categories.

This structure is useful for multi-class blood cell classification because the convolutional layers progressively learn visual features such as colour, texture, shape, and cell morphology, while the residual connections help preserve useful information and improve gradient flow through the deeper network.

## 2. Loading the BloodMNIST Dataset

The BloodMNIST dataset is loaded in `A3code.py` using the `LoadDataBloodMNIST()` function. This function uses the MedMNIST metadata dictionary to obtain the correct BloodMNIST dataset class, number of input channels, number of classes, and class label information. 

1. The official training, validation, and test splits are loaded separately so that the model can be trained, tuned, and finally evaluated using distinct datasets.

2. The loading processthen applies the required preprocessing before the images are passed into the neural network. Each image is converted into a PyTorch tensor so that it follows the expected `[channels, height, width]` format for convolutional layers, and normalization is applied to scale the pixel values into a range that is more suitable for stable neural network training. 

3. Finally, each dataset split is wrapped in a PyTorch `DataLoader`, allowing the training and evaluation functions to process the data in mini-batches rather than loading the full dataset at once.


## 3. Training Using CrossEntropyLoss and Gradient Updating Using Adam Optimizer

### 3.1 CrossEntropyLoss

`nn.CrossEntropyLoss` was used as the loss function because BloodMNIST is an 8-class classification problem where each image belongs to one blood cell class. The PyTorch documentation describes CrossEntropyLoss as a criterion that compares the model's input logits against the target class labels, making it suitable for classification tasks with `C` classes [5]. This also means that the ResNet18 model should output raw, unnormalised logits rather than manually applying a final softmax layer, since CrossEntropyLoss internally combines the behaviour of `LogSoftmax` and negative log-likelihood loss.

For one sample `n`, where `x` represents the raw output logits and `y_n` is the correct class label, the class-index form of the loss can be written as:

$$
\ell_n = -\log\left(\frac{\exp(x_{n,y_n})}{\sum_{c=1}^{C}\exp(x_{n,c})}\right)
$$

where `C` is the number of classes. With the default mean reduction, the mini-batch loss is averaged across `N` samples:

$$
L = \frac{1}{N}\sum_{n=1}^{N}\ell_n
$$

In `A3code.py`, the loss function is implemented using:

```python
criterion = nn.CrossEntropyLoss()
```

During training and evaluation, the model first produces class logits from the input images, then the criterion compares those logits with the true labels:

```python
outputs = model(images)
loss = criterion(outputs, labels)
```

The labels are converted into integer class indices using `.long()`, which matches the target format expected by CrossEntropyLoss. The calculated loss then becomes the main training signal for backpropagation:

```python
loss.backward()
```

This step calculates how much each trainable parameter contributed to the classification error, allowing the optimizer to update the network weights.

### 3.2 Adam Optimizer

Adam, or Adaptive Moment Estimation, was used as the optimizer for gradient-based weight updates. Kingma and Ba describe Adam as a first-order stochastic optimization method that uses adaptive estimates of lower-order moments of the gradients [4]. In practical terms, Adam tracks both the moving average of the gradients and the moving average of the squared gradients, allowing each parameter to receive an adaptive update size rather than relying on a single fixed gradient scale across the whole network.

At training step `t`, Adam first uses the gradient of the current loss with respect to the model parameters:

$$
g_t = \nabla_{\theta} L_t(\theta_{t-1})
$$

It then updates the first moment estimate, which behaves like momentum by tracking the recent direction of the gradients:

$$
m_t = \beta_1m_{t-1} + (1 - \beta_1)g_t
$$

Adam also updates the second raw moment estimate, which tracks the recent magnitude of the squared gradients:

$$
v_t = \beta_2v_{t-1} + (1 - \beta_2)g_t^2
$$

Because both moment estimates are initialised at zero, Adam applies bias correction:

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}
$$

$$
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

The final parameter update is then:

$$
\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

where $\alpha$ is the learning rate and $\epsilon$ is a small constant used for numerical stability. This optimizer was appropriate for this project because the ResNet18 model contains many convolutional and fully connected parameters, and Adam can adapt the update size for different parameters based on their recent gradient behaviour.

In `A3code.py`, Adam is implemented using:

```python
optimizer = optim.Adam(model.parameters(), lr=learn_rate)
```

Inside the training loop, the gradient update sequence is:

```python
optimizer.zero_grad()
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
optimizer.step()
```

`optimizer.zero_grad()` clears old gradients from the previous mini-batch, `loss.backward()` computes the new gradients from CrossEntropyLoss, gradient clipping limits very large gradients, and `optimizer.step()` applies the Adam update to the model weights. Therefore, CrossEntropyLoss defines the classification error signal, while Adam uses the resulting gradients to progressively adjust the ResNet18 parameters during training.



## 4. Hyperparameter Tuning Selection

Hyperparameter tuning was done in two stages. The training set was used to update the ResNet18 weights, while the validation set was used to decide which hyperparameters were working best. The test set was kept separate so it could be used for the final model evaluation only.

The first stage tested channel size and learning rate while keeping the training length fixed at 50 epochs. Channel size changes how many feature maps the residual network can learn, while learning rate controls the size of each Adam optimizer update. This produced Models 1-9.

| Model Group | Channel Sizes | Learning Rate | Epochs |
|---|---:|---:|---:|
| Models 1-3 | `[8, 16, 32, 64]`, `[32, 64, 128, 256]`, `[64, 128, 256, 512]` | `1e-3` | 50 |
| Models 4-6 | `[8, 16, 32, 64]`, `[32, 64, 128, 256]`, `[64, 128, 256, 512]` | `1e-4` | 50 |
| Models 7-9 | `[8, 16, 32, 64]`, `[32, 64, 128, 256]`, `[64, 128, 256, 512]` | `1e-5` | 50 |

The main selection metrics were validation accuracy, validation loss, and the train-validation accuracy gap. I also checked final validation performance, training time, and the loss curves to see whether the model was still improving, plateauing, or starting to overfit.

| Selection Metric | Reason for use |
|---|---|
| Highest validation accuracy | Shows the best validation classification performance reached by the model. |
| Lowest validation loss | Shows the lowest validation error and confidence penalty. |
| Final validation accuracy and loss | Checks whether the final saved model was still performing well. |
| Train-validation accuracy gap | Helps identify overfitting when training accuracy is much higher than validation accuracy. |
| Training time | Shows whether extra training gave enough benefit to justify the time cost. |
| Validation loss stability / loss curves | Shows whether training was stable, plateauing, or degrading near the end. |

After the best channel size and learning rate were chosen from Models 1-9, the second stage tuned the number of epochs. For this stage, the selected channel size and learning rate were kept fixed, and only the training duration was changed in Models 10-13.

| Model | Channel Sizes | Learning Rate | Epochs Tested |
|---|---:|---:|---:|
| Model 10 | Selected from stage 1 | Selected from stage 1 | 10 |
| Model 11 | Selected from stage 1 | Selected from stage 1 | 25 |
| Model 12 | Selected from stage 1 | Selected from stage 1 | 30 |
| Model 13 | Selected from stage 1 | Selected from stage 1 | 45 |

This second stage was used to find a training length that was long enough for the model to learn useful image features, but not so long that validation performance started to fall.



## 5. Results of Hyperparameter Tuning

The first tuning stage showed that `1e-3` was the most effective learning rate overall. The lower learning rates, `1e-4` and `1e-5`, trained more slowly and generally produced weaker validation performance. Within the `1e-3` group, Model 2 gave the strongest peak validation result, using channel sizes `[32, 64, 128, 256]`.

| Model | Channels | Learning Rate | Epochs | Highest Val Acc (%) | Lowest Val Loss | Final Val Acc (%) | Final Val Loss | Final Gap (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Model 1 | `[8, 16, 32, 64]` | `1e-3` | 50 | 91.36 | 0.2843 | 88.49 | 0.4317 | 8.10 |
| **Model 2** | **`[32, 64, 128, 256]`** | **`1e-3`** | **50** | **93.69** | **0.2441** | 88.73 | 0.5325 | 9.84 |
| Model 3 | `[64, 128, 256, 512]` | `1e-3` | 50 | 92.76 | 0.2704 | 92.41 | 0.3490 | 6.40 |

Model 2 was therefore used as the basis for the epoch refinement stage. However, even though it reached the best peak validation accuracy, its final validation accuracy dropped to 88.73% and its final validation loss increased to 0.5325 by epoch 50. This suggested that the channel size and learning rate were strong, but 50 epochs was likely longer than needed.

| Model | Channels | Learning Rate | Epochs | Highest Val Acc (%) | Lowest Val Loss | Final Val Acc (%) | Final Val Loss | Final Gap (%) | Training Time |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Model 10 | `[32, 64, 128, 256]` | `1e-3` | 10 | 88.67 | 0.3803 | 88.67 | 0.3803 | 5.05 | 0 min 45 sec |
| **Model 11** | **`[32, 64, 128, 256]`** | **`1e-3`** | **25** | **91.82** | **0.3003** | **91.30** | **0.3044** | **5.60** | **2 min 3 sec** |
| Model 12 | `[32, 64, 128, 256]` | `1e-3` | 30 | 91.41 | 0.3090 | 89.02 | 0.4744 | 8.32 | 2 min 43 sec |
| Model 13 | `[32, 64, 128, 256]` | `1e-3` | 45 | 91.88 | 0.2932 | 90.54 | 0.3576 | 7.75 | 4 min 17 sec |

Model 11 was selected as the final model. It did not have the single highest peak value, but it had the best balance of final validation accuracy, final validation loss, train-validation gap, and training time. Model 10 was undertrained, while Models 12 and 13 showed signs that longer training did not improve the final saved model enough to justify the extra time.

The full model comparison figure is shown below, and the complete outputs for each model are provided in Appendix A.

![Model comparison table for Models 1 to 13](Pre-Trained%20Models/Model%20Comparison%20Figure1%20to%2013.png)


The final hyperparameter set combined the strongest channel-size and learning-rate setting from Model 2 with the best epoch count found in Model 11.

| Final Hyperparameter | Selected Value | Source Model |
|---|---:|---:|
| Channel sizes | `[32, 64, 128, 256]` | Model 2 |
| Learning rate | `1e-3` | Model 2 |
| Epochs | `25` | Model 11 |




## 6. Discussion of Hyperparameter Tuning

As shown in the full comparison figure in Section 5, `1e-3` was the strongest learning rate tested. With channel size `[32, 64, 128, 256]`, Model 2 reached 93.69% validation accuracy and 0.2441 validation loss, while `1e-4` and `1e-5` were lower at 89.84% and 83.18% validation accuracy. This suggests the smaller learning rates were too slow for the 50-epoch limit. Adam adapts updates using moving averages of gradients and squared gradients, but the learning rate still controls the overall update size [4].

The `[32, 64, 128, 256]` channel setting gave the best peak validation result in the first search. The smaller `[8, 16, 32, 64]` model likely had too little capacity to capture fine blood-cell features, while the larger `[64, 128, 256, 512]` model required more computation without giving the best peak validation accuracy or loss. This supports the idea that hyperparameter tuning should balance model capacity and generalisation rather than simply choosing the largest model [3]. The relevant model summaries and loss curves are included in Appendix A for Models 1-9.

The epoch refinement results also show that longer training was not automatically better. Model 2 performed strongly at its peak, but by epoch 50 its final validation accuracy had dropped to 88.73% and its validation loss increased to 0.5325. Model 11, trained for 25 epochs with the same channel size and learning rate, gave the best practical balance: 91.30% final validation accuracy, 0.3044 final validation loss, a 5.60% train-validation gap, and 2 min 3 sec training time. The Appendix A figures for Models 10-13 show this epoch comparison in more detail, including the loss curves used to judge convergence and overfitting.



## 7. Future Study and Improvement

I personally ran out of time to experiment with the model further but I would want to look into other NN sturctures for image classification such as:

1. Transformer Revolution - hardwired rules that assume nearby pixels are related, making them highly efficient and accurate even with smaller amounts of training data
2. ConvNets - looking at the entire image at once without any pre-set rules, allowing them to achieve unmatched accuracy when trained on massive datasets


I would also like to learn what exact hyperparamters they chose and learning method for the official MedMNIST study as they were able to achieve an overall accuracy of 99.8% classification [2].

<!-- APPENDIX_MODEL_METRICS_START -->

## 7. Appendix

### Appendix A: Model Output Metrics

This appendix contains compact versions of the saved metric figures for Models 1-13. The layout is designed so that two models can fit on one printed PDF page, with each model shown as a 2x2 figure grid.

<style>
.model-pair-page {
  break-after: page;
  page-break-after: always;
}
.model-block {
  margin: 0 0 10px 0;
}
.model-block h3 {
  margin: 4px 0 6px 0;
  font-size: 14px;
}
.model-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 5px 8px;
}
.figure-box {
  break-inside: avoid;
  page-break-inside: avoid;
  text-align: center;
}
.figure-box img {
  max-width: 100%;
  max-height: 165px;
  object-fit: contain;
}
.figure-caption {
  font-size: 9px;
  line-height: 1.15;
  margin-top: 2px;
}
@media print {
  .model-block h3 {
    font-size: 12px;
  }
  .figure-box img {
    max-height: 150px;
  }
  .figure-caption {
    font-size: 8px;
  }
}
</style>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 1 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%201/Model%20Outputs%20-%20Model%201/Classification_Metrics_Test_Set.png" alt="Figure A1.1 - Classification Metrics for Model 1">
<div class="figure-caption"><strong>Figure A1.1.</strong> Classification Metrics for Model 1.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%201/Model%20Outputs%20-%20Model%201/Confusion_Matrix_Test_Set.png" alt="Figure A1.2 - Confusion Matrix for Model 1">
<div class="figure-caption"><strong>Figure A1.2.</strong> Confusion Matrix for Model 1.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%201/Model%20Outputs%20-%20Model%201/Training_Validation_Loss.png" alt="Figure A1.3 - Training Validation for Model 1">
<div class="figure-caption"><strong>Figure A1.3.</strong> Training Validation for Model 1.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%201/Model%20Outputs%20-%20Model%201/Model_Summary_Metrics.png" alt="Figure A1.4 - Model Summary for Model 1">
<div class="figure-caption"><strong>Figure A1.4.</strong> Model Summary for Model 1.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 2 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%202/Model%20Outputs%20-%20Model%202/Classification_Metrics_Test_Set.png" alt="Figure A2.1 - Classification Metrics for Model 2">
<div class="figure-caption"><strong>Figure A2.1.</strong> Classification Metrics for Model 2.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%202/Model%20Outputs%20-%20Model%202/Confusion_Matrix_Test_Set.png" alt="Figure A2.2 - Confusion Matrix for Model 2">
<div class="figure-caption"><strong>Figure A2.2.</strong> Confusion Matrix for Model 2.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%202/Model%20Outputs%20-%20Model%202/Training_Validation_Loss.png" alt="Figure A2.3 - Training Validation for Model 2">
<div class="figure-caption"><strong>Figure A2.3.</strong> Training Validation for Model 2.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%202/Model%20Outputs%20-%20Model%202/Model_Summary_Metrics.png" alt="Figure A2.4 - Model Summary for Model 2">
<div class="figure-caption"><strong>Figure A2.4.</strong> Model Summary for Model 2.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 3 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%203/Model%20Outputs%20-%20Model%203/Classification_Metrics_Test_Set.png" alt="Figure A3.1 - Classification Metrics for Model 3">
<div class="figure-caption"><strong>Figure A3.1.</strong> Classification Metrics for Model 3.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%203/Model%20Outputs%20-%20Model%203/Confusion_Matrix_Test_Set.png" alt="Figure A3.2 - Confusion Matrix for Model 3">
<div class="figure-caption"><strong>Figure A3.2.</strong> Confusion Matrix for Model 3.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%203/Model%20Outputs%20-%20Model%203/Training_Validation_Loss.png" alt="Figure A3.3 - Training Validation for Model 3">
<div class="figure-caption"><strong>Figure A3.3.</strong> Training Validation for Model 3.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%203/Model%20Outputs%20-%20Model%203/Model_Summary_Metrics.png" alt="Figure A3.4 - Model Summary for Model 3">
<div class="figure-caption"><strong>Figure A3.4.</strong> Model Summary for Model 3.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 4 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%204/Model%20Outputs%20-%20Model%204/Classification_Metrics_Test_Set.png" alt="Figure A4.1 - Classification Metrics for Model 4">
<div class="figure-caption"><strong>Figure A4.1.</strong> Classification Metrics for Model 4.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%204/Model%20Outputs%20-%20Model%204/Confusion_Matrix_Test_Set.png" alt="Figure A4.2 - Confusion Matrix for Model 4">
<div class="figure-caption"><strong>Figure A4.2.</strong> Confusion Matrix for Model 4.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%204/Model%20Outputs%20-%20Model%204/Training_Validation_Loss.png" alt="Figure A4.3 - Training Validation for Model 4">
<div class="figure-caption"><strong>Figure A4.3.</strong> Training Validation for Model 4.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%204/Model%20Outputs%20-%20Model%204/Model_Summary_Metrics.png" alt="Figure A4.4 - Model Summary for Model 4">
<div class="figure-caption"><strong>Figure A4.4.</strong> Model Summary for Model 4.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 5 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%205/Model%20Outputs%20-%20Model%205/Classification_Metrics_Test_Set.png" alt="Figure A5.1 - Classification Metrics for Model 5">
<div class="figure-caption"><strong>Figure A5.1.</strong> Classification Metrics for Model 5.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%205/Model%20Outputs%20-%20Model%205/Confusion_Matrix_Test_Set.png" alt="Figure A5.2 - Confusion Matrix for Model 5">
<div class="figure-caption"><strong>Figure A5.2.</strong> Confusion Matrix for Model 5.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%205/Model%20Outputs%20-%20Model%205/Training_Validation_Loss.png" alt="Figure A5.3 - Training Validation for Model 5">
<div class="figure-caption"><strong>Figure A5.3.</strong> Training Validation for Model 5.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%205/Model%20Outputs%20-%20Model%205/Model_Summary_Metrics.png" alt="Figure A5.4 - Model Summary for Model 5">
<div class="figure-caption"><strong>Figure A5.4.</strong> Model Summary for Model 5.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 6 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%206/Model%20Outputs%20-%20Model%206/Classification_Metrics_Test_Set.png" alt="Figure A6.1 - Classification Metrics for Model 6">
<div class="figure-caption"><strong>Figure A6.1.</strong> Classification Metrics for Model 6.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%206/Model%20Outputs%20-%20Model%206/Confusion_Matrix_Test_Set.png" alt="Figure A6.2 - Confusion Matrix for Model 6">
<div class="figure-caption"><strong>Figure A6.2.</strong> Confusion Matrix for Model 6.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%206/Model%20Outputs%20-%20Model%206/Training_Validation_Loss.png" alt="Figure A6.3 - Training Validation for Model 6">
<div class="figure-caption"><strong>Figure A6.3.</strong> Training Validation for Model 6.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%206/Model%20Outputs%20-%20Model%206/Model_Summary_Metrics.png" alt="Figure A6.4 - Model Summary for Model 6">
<div class="figure-caption"><strong>Figure A6.4.</strong> Model Summary for Model 6.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 7 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%207/Model%20Outputs%20-%20Model%207/Classification_Metrics_Test_Set.png" alt="Figure A7.1 - Classification Metrics for Model 7">
<div class="figure-caption"><strong>Figure A7.1.</strong> Classification Metrics for Model 7.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%207/Model%20Outputs%20-%20Model%207/Confusion_Matrix_Test_Set.png" alt="Figure A7.2 - Confusion Matrix for Model 7">
<div class="figure-caption"><strong>Figure A7.2.</strong> Confusion Matrix for Model 7.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%207/Model%20Outputs%20-%20Model%207/Training_Validation_Loss.png" alt="Figure A7.3 - Training Validation for Model 7">
<div class="figure-caption"><strong>Figure A7.3.</strong> Training Validation for Model 7.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%207/Model%20Outputs%20-%20Model%207/Model_Summary_Metrics.png" alt="Figure A7.4 - Model Summary for Model 7">
<div class="figure-caption"><strong>Figure A7.4.</strong> Model Summary for Model 7.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 8 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%208/Model%20Outputs%20-%20Model%208/Classification_Metrics_Test_Set.png" alt="Figure A8.1 - Classification Metrics for Model 8">
<div class="figure-caption"><strong>Figure A8.1.</strong> Classification Metrics for Model 8.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%208/Model%20Outputs%20-%20Model%208/Confusion_Matrix_Test_Set.png" alt="Figure A8.2 - Confusion Matrix for Model 8">
<div class="figure-caption"><strong>Figure A8.2.</strong> Confusion Matrix for Model 8.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%208/Model%20Outputs%20-%20Model%208/Training_Validation_Loss.png" alt="Figure A8.3 - Training Validation for Model 8">
<div class="figure-caption"><strong>Figure A8.3.</strong> Training Validation for Model 8.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%208/Model%20Outputs%20-%20Model%208/Model_Summary_Metrics.png" alt="Figure A8.4 - Model Summary for Model 8">
<div class="figure-caption"><strong>Figure A8.4.</strong> Model Summary for Model 8.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 9 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%209/Model%20Outputs%20-%20Model%209/Classification_Metrics_Test_Set.png" alt="Figure A9.1 - Classification Metrics for Model 9">
<div class="figure-caption"><strong>Figure A9.1.</strong> Classification Metrics for Model 9.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%209/Model%20Outputs%20-%20Model%209/Confusion_Matrix_Test_Set.png" alt="Figure A9.2 - Confusion Matrix for Model 9">
<div class="figure-caption"><strong>Figure A9.2.</strong> Confusion Matrix for Model 9.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%209/Model%20Outputs%20-%20Model%209/Training_Validation_Loss.png" alt="Figure A9.3 - Training Validation for Model 9">
<div class="figure-caption"><strong>Figure A9.3.</strong> Training Validation for Model 9.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%209/Model%20Outputs%20-%20Model%209/Model_Summary_Metrics.png" alt="Figure A9.4 - Model Summary for Model 9">
<div class="figure-caption"><strong>Figure A9.4.</strong> Model Summary for Model 9.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 10 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2010/Model%20Outputs%20-%20Model%2010/Classification_Metrics_Test_Set.png" alt="Figure A10.1 - Classification Metrics for Model 10">
<div class="figure-caption"><strong>Figure A10.1.</strong> Classification Metrics for Model 10.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2010/Model%20Outputs%20-%20Model%2010/Confusion_Matrix_Test_Set.png" alt="Figure A10.2 - Confusion Matrix for Model 10">
<div class="figure-caption"><strong>Figure A10.2.</strong> Confusion Matrix for Model 10.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2010/Model%20Outputs%20-%20Model%2010/Training_Validation_Loss.png" alt="Figure A10.3 - Training Validation for Model 10">
<div class="figure-caption"><strong>Figure A10.3.</strong> Training Validation for Model 10.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2010/Model%20Outputs%20-%20Model%2010/Model_Summary_Metrics.png" alt="Figure A10.4 - Model Summary for Model 10">
<div class="figure-caption"><strong>Figure A10.4.</strong> Model Summary for Model 10.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 11 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2011/Model%20Outputs%20-%20Model%2011/Classification_Metrics_Test_Set.png" alt="Figure A11.1 - Classification Metrics for Model 11">
<div class="figure-caption"><strong>Figure A11.1.</strong> Classification Metrics for Model 11.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2011/Model%20Outputs%20-%20Model%2011/Confusion_Matrix_Test_Set.png" alt="Figure A11.2 - Confusion Matrix for Model 11">
<div class="figure-caption"><strong>Figure A11.2.</strong> Confusion Matrix for Model 11.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2011/Model%20Outputs%20-%20Model%2011/Training_Validation_Loss.png" alt="Figure A11.3 - Training Validation for Model 11">
<div class="figure-caption"><strong>Figure A11.3.</strong> Training Validation for Model 11.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2011/Model%20Outputs%20-%20Model%2011/Model_Summary_Metrics.png" alt="Figure A11.4 - Model Summary for Model 11">
<div class="figure-caption"><strong>Figure A11.4.</strong> Model Summary for Model 11.</div>
</div>
</div>
</div>
<div class="model-block">
<h3>Model 12 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2012/Model%20Outputs%20-%20Model%2012/Classification_Metrics_Test_Set.png" alt="Figure A12.1 - Classification Metrics for Model 12">
<div class="figure-caption"><strong>Figure A12.1.</strong> Classification Metrics for Model 12.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2012/Model%20Outputs%20-%20Model%2012/Confusion_Matrix_Test_Set.png" alt="Figure A12.2 - Confusion Matrix for Model 12">
<div class="figure-caption"><strong>Figure A12.2.</strong> Confusion Matrix for Model 12.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2012/Model%20Outputs%20-%20Model%2012/Training_Validation_Loss.png" alt="Figure A12.3 - Training Validation for Model 12">
<div class="figure-caption"><strong>Figure A12.3.</strong> Training Validation for Model 12.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2012/Model%20Outputs%20-%20Model%2012/Model_Summary_Metrics.png" alt="Figure A12.4 - Model Summary for Model 12">
<div class="figure-caption"><strong>Figure A12.4.</strong> Model Summary for Model 12.</div>
</div>
</div>
</div>
</div>


<div class="model-pair-page">
<div class="model-block">
<h3>Model 13 Metrics</h3>
<div class="model-grid">
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2013/Model%20Outputs%20-%20Model%2013/Classification_Metrics_Test_Set.png" alt="Figure A13.1 - Classification Metrics for Model 13">
<div class="figure-caption"><strong>Figure A13.1.</strong> Classification Metrics for Model 13.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2013/Model%20Outputs%20-%20Model%2013/Confusion_Matrix_Test_Set.png" alt="Figure A13.2 - Confusion Matrix for Model 13">
<div class="figure-caption"><strong>Figure A13.2.</strong> Confusion Matrix for Model 13.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2013/Model%20Outputs%20-%20Model%2013/Training_Validation_Loss.png" alt="Figure A13.3 - Training Validation for Model 13">
<div class="figure-caption"><strong>Figure A13.3.</strong> Training Validation for Model 13.</div>
</div>
<div class="figure-box">
<img src="Pre-Trained%20Models/Model%2013/Model%20Outputs%20-%20Model%2013/Model_Summary_Metrics.png" alt="Figure A13.4 - Model Summary for Model 13">
<div class="figure-caption"><strong>Figure A13.4.</strong> Model Summary for Model 13.</div>
</div>
</div>
</div>
</div>


<!-- APPENDIX_MODEL_METRICS_END -->
